# setup + imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

from gensim.models import Word2Vec

In [2]:
train_data = pd.read_csv('train_data.csv')
test_data = pd.read_csv('test_data.csv')
unlabeled_data = pd.read_csv('unlabeled_data.csv')

In [3]:
X = train_data['text']
y = train_data['label']

# subtask 1

In [4]:
words = []
for row in unlabeled_data['text']:
    cuv = row.split(' ')
    for w in cuv:
        if w not in words:
            words.append(w)

subtask1 = pd.DataFrame({
    'subtaskID': 1,
    'datapointID': 0,
    'answer': int(len(words))
}, index=[0])
    

# data exploration

In [5]:
train_words = set(' '.join(train_data['text']).split())
unl_words = set(' '.join(unlabeled_data['text']).split())
test_words = ' '.join(test_data['text']).split()

unseen = []
for w in test_words:
  if w not in train_words:
    unseen.append(w)
print('test words never seen in train:', round(len(unseen) / len(test_words), 3))

found = 0
for w in unseen:
  if w in unl_words:
    found += 1
print('of those, found in unlabeled:', round(found / max(1, len(unseen)), 3))

test words never seen in train: 0.085
of those, found in unlabeled: 1.0


# validare

In [6]:
def evaluate(vectorizer, model, X, y):
  kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

  scores = []
  for fit_rows, val_rows in kf.split(X, y):
    X_fit = X.iloc[fit_rows]
    y_fit = y.iloc[fit_rows]
    X_val = X.iloc[val_rows]
    y_val = y.iloc[val_rows]

    X_fit = vectorizer.fit_transform(X_fit)
    X_val = vectorizer.transform(X_val)

    model.fit(X_fit, y_fit)
    preds = model.predict(X_val)
    score = f1_score(y_val, preds, average='macro')

    scores.append(score)
  return np.mean(scores)

# model

In [7]:
text = pd.concat([unlabeled_data['text'], train_data['text'], test_data['text']])
sentences = [t.split() for t in text]
w2v = Word2Vec(sentences, vector_size=100, window=3, min_count=2, sg=1, epochs=50, seed=0, workers=1)

def texvec(t):
    vecs = []
    for w in t.split():
        if w in w2v.wv:
            vecs.append(w2v.wv[w])
    if len(vecs) == 0:
        return np.zeros(w2v.vector_size)
    return np.mean(vecs, axis=0)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [8]:
W = pd.DataFrame([texvec(t) for t in X])
W_test = pd.DataFrame([texvec(t) for t in test_data['text']])

In [9]:
'''
options = {
    'basic': dict(),
    'sublinear': dict(sublinear_tf=True),
    'words': dict(ngram_range=(1, 2), sublinear_tf=True)
}

results = {}
for name in options.keys():
    param = options[name]
    vectorizer = TfidfVectorizer(**param)
    model = LogisticRegression()
    results[name] = evaluate(vectorizer, model, X, y)

print(results)

bestname = ""
bestr = 0
for name in results.keys():
  r = results[name]
  if r > bestr:
    bestr = r
    bestname = name

print(bestname, bestr)

for C in [1, 10, 100]:
    print('C =', C, evaluate(TfidfVectorizer(**options[bestname]), LogisticRegression(C=C, max_iter=2000), X, y))
'''

'\noptions = {\n    \'basic\': dict(),\n    \'sublinear\': dict(sublinear_tf=True),\n    \'words\': dict(ngram_range=(1, 2), sublinear_tf=True)\n}\n\nresults = {}\nfor name in options.keys():\n    param = options[name]\n    vectorizer = TfidfVectorizer(**param)\n    model = LogisticRegression()\n    results[name] = evaluate(vectorizer, model, X, y)\n\nprint(results)\n\nbestname = ""\nbestr = 0\nfor name in results.keys():\n  r = results[name]\n  if r > bestr:\n    bestr = r\n    bestname = name\n\nprint(bestname, bestr)\n\nfor C in [1, 10, 100]:\n    print(\'C =\', C, evaluate(TfidfVectorizer(**options[bestname]), LogisticRegression(C=C, max_iter=2000), X, y))\n'

In [10]:
'''vectorizer = TfidfVectorizer(**options[bestname])
model = LogisticRegression(C=10, max_iter=2000)

X_full = vectorizer.fit_transform(X)
model.fit(X_full, y)
preds = model.predict(vectorizer.transform(test_data["text"]))'''

'vectorizer = TfidfVectorizer(**options[bestname])\nmodel = LogisticRegression(C=10, max_iter=2000)\n\nX_full = vectorizer.fit_transform(X)\nmodel.fit(X_full, y)\npreds = model.predict(vectorizer.transform(test_data["text"]))'

In [11]:
model = LogisticRegression(C=10, max_iter=2000).fit(W, y)
preds = model.predict(W_test)
print(pd.Series(preds).value_counts(normalize=True))  

2    0.2555
3    0.2500
0    0.2485
1    0.2460
Name: proportion, dtype: float64


In [12]:
subtask2 = pd.DataFrame({
    'subtaskID': 2,
    'datapointID': test_data['id'],
    'answer': preds
})
    

In [13]:
submission = pd.concat([subtask1, subtask2], ignore_index=True)
submission.to_csv('submission.csv', index=False)